In [1]:
library(Signac)
library(Seurat)
library(Matrix)
library(EnsDb.Hsapiens.v75)


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: ensembldb

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    

In [2]:
library(BSgenome.Hsapiens.UCSC.hg19)

Loading required package: BSgenome

Loading required package: Biostrings

Loading required package: XVector


Attaching package: 'Biostrings'


The following object is masked from 'package:base':

    strsplit


Loading required package: BiocIO

Loading required package: rtracklayer



# Load ATAC dataset

In [3]:
filepath <- "GSE129785_scATAC-Hematopoiesis-CD34"

# Load peak matrix
peaks <- read.table(paste0(filepath, ".peaks.txt.gz"), header = TRUE)
cells <- read.table(paste0(filepath, ".cell_barcodes.txt.gz"), header = TRUE)
mtx <- readMM(paste0(filepath, ".mtx.gz"))

mtx <- as(mtx, "CsparseMatrix")

In [4]:
head(peaks$Feature)

[1] "chr1_10238_10738"   "chr1_115476_115976" "chr1_236856_237356"
[4] "chr1_237511_238011" "chr1_240801_241301" "chr1_250084_250584"

In [5]:
head(cells$Barcodes)

[1] "CAAGAAAGTCAAGACG-1" "AAAGGGCAGTACCCAT-1" "CACAACATCGTGGGTC-1"
[4] "TTACCCGTCTGATCCC-1" "CTTTGCGGTACGGTTT-1" "GGTTGCGCAGGGAGTT-1"

In [6]:
library(GenomicRanges)
library(stringr)

peaks <- read.table(
  paste0(filepath, ".peaks.txt.gz"),
  header = TRUE
)

coords <- str_split_fixed(peaks$Feature, "_", 3)

gr <- GRanges(
  seqnames = coords[,1],
  ranges = IRanges(
    start = as.numeric(coords[,2]),
    end   = as.numeric(coords[,3])
  )
)

In [7]:
seqlevelsStyle(gr) <- "UCSC"

In [8]:
head(gr)

GRanges object with 6 ranges and 0 metadata columns:
      seqnames        ranges strand
         <Rle>     <IRanges>  <Rle>
  [1]     chr1   10238-10738      *
  [2]     chr1 115476-115976      *
  [3]     chr1 236856-237356      *
  [4]     chr1 237511-238011      *
  [5]     chr1 240801-241301      *
  [6]     chr1 250084-250584      *
  -------
  seqinfo: 23 sequences from an unspecified genome; no seqlengths

In [9]:
rownames(mtx) <- paste0(
  seqnames(gr), ":", start(gr), "-", end(gr)
)

colnames(mtx) <- make.unique(cells$Barcodes)

In [10]:
# Create Seurat object
chrom_assay <- CreateChromatinAssay(
  counts = mtx,
  ranges = gr,
  fragments = "GSM3722029_CD34_Progenitors_Rep1_fragments.tsv.gz",
  genome = "hg19"
)

Computing hash



In [11]:
obj <- CreateSeuratObject(
  counts = chrom_assay,
  assay = "ATAC"
)

In [12]:
class(obj)

[1] "Seurat"
attr(,"package")
[1] "SeuratObject"

In [13]:
obj

An object of class Seurat 
571389 features across 18489 samples within 1 assay 
Active assay: ATAC (571389 features, 0 variable features)
 2 layers present: counts, data

In [14]:
DefaultAssay(obj) <- "ATAC"

# Gene annotation

In [15]:
annotations <- GetGRangesFromEnsDb(EnsDb.Hsapiens.v75)

seqlevelsStyle(annotations) <- "UCSC"
genome(annotations) <- "hg19"

Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warn

In [16]:
seqlevels(gr)

[1] "chr1"  "chr2"  "chr3"  "chr4"  "chr5"  "chr6"  "chr7"  "chr8"  "chr9" 
[10] "chr10" "chr11" "chr12" "chr13" "chr14" "chr15" "chr16" "chr17" "chr18"
[19] "chr19" "chr20" "chr21" "chr22" "chrX"

In [17]:
library(GenomeInfoDb)

standard_chr <- paste0("chr", c(1:22, "X"))

annotations <- keepSeqlevels(
  annotations,
  standard_chr,
  pruning.mode = "coarse"
)

gr <- keepSeqlevels(
  gr,
  standard_chr,
  pruning.mode = "coarse"
)

obj <- keepSeqlevels(
  obj,
  standard_chr,
  pruning.mode = "coarse"
)

In [18]:
Annotation(obj) <- annotations

In [19]:
seqlevelsStyle(obj) <- "UCSC"
seqlevels(obj)
seqlevels(annotations)[1:5]
seqlevels(gr)[1:5]

[1] "chr1"  "chr2"  "chr3"  "chr4"  "chr5"  "chr6"  "chr7"  "chr8"  "chr9" 
[10] "chr10" "chr11" "chr12" "chr13" "chr14" "chr15" "chr16" "chr17" "chr18"
[19] "chr19" "chr20" "chr21" "chr22" "chrX"

[1] "chr1" "chr2" "chr3" "chr4" "chr5"

[1] "chr1" "chr2" "chr3" "chr4" "chr5"

# Peak --> gene links

In [20]:
obj <- RegionStats(obj, genome = BSgenome.Hsapiens.UCSC.hg19)

In [22]:
gene.activities <- GeneActivity(obj)

Extracting gene coordinates

Extracting reads overlapping genomic regions



In [23]:
obj[["RNA"]] <- CreateAssayObject(counts = gene.activities)
DefaultAssay(obj) <- "RNA"
obj <- NormalizeData(obj)
obj <- FindVariableFeatures(obj)
DefaultAssay(obj) <- "ATAC"

Warning message:
"Non-unique features (rownames) present in the input matrix, making unique"


In [27]:
obj <- FindTopFeatures(obj, min.cutoff = 'q0')
top_peaks <- VariableFeatures(obj)
obj <- subset(obj, features = top_peaks)

In [36]:
keep_peaks <- rowSums(GetAssayData(obj, layer="counts") > 0) > 250
obj <- subset(obj, features = names(keep_peaks[keep_peaks]))

In [38]:
obj$ATAC

ChromatinAssay data with 139723 features for 18489 cells
Variable features: 139723 
Genome: hg19 
Annotation present: TRUE 
Motifs present: FALSE 
Fragment files: 1 

In [39]:
gene.activities <- GeneActivity(obj)
obj[["RNA"]] <- CreateAssayObject(counts = gene.activities)
DefaultAssay(obj) <- "RNA"
obj <- NormalizeData(obj)
obj <- FindVariableFeatures(obj)
DefaultAssay(obj) <- "ATAC"

Extracting gene coordinates

Extracting reads overlapping genomic regions

Warning message:
"Non-unique features (rownames) present in the input matrix, making unique"


In [40]:
obj <- LinkPeaks(
  obj,
  peak.assay = "ATAC",
  expression.assay = "RNA",
  distance = 50000
)

peak_gene <- ClosestFeature(obj)

Found gene coordinates for 19554 genes

Testing 19557 genes and 98744 peaks



In [29]:
Assays(obj)


[1] "ATAC"

# Add TF motifs

In [ ]:
library(JASPAR2020)
library(TFBSTools)
library(BSgenome.Hsapiens.UCSC.hg19)

pfm <- getMatrixSet(
  x = JASPAR2020,
  opts = list(species = 9606)
)

obj <- AddMotifs(
  object = obj,
  genome = BSgenome.Hsapiens.UCSC.hg19,
  pfm = pfm
)